In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ds_310_project_2_fall_2024_path = kagglehub.competition_download('ds-310-project-2-fall-2024')
mattviana_tuner0_path = kagglehub.dataset_download('mattviana/tuner0')
mattviana_notamodel_other_default_1_path = kagglehub.model_download('mattviana/notamodel/Other/default/1')

print('Data source import complete.')


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import class_weight
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, add, Activation, Lambda, Attention
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from keras_tuner import HyperParameters, RandomSearch
import json
import os
import joblib


In [ ]:
import shutil

shutil.make_archive('/kaggle/working/hyperparam_tuning', 'zip', '/kaggle/working/hyperparam_tuning')


In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_df = pd.read_csv('/kaggle/input/ds-310-project-2-fall-2024/train_data.csv')

X = train_df.drop(['Name', 'Class'], axis=1)
y = train_df['Class']

y = LabelEncoder().fit_transform(y)

categorical_cols = ['Sector']
numerical_cols = X.columns.difference(categorical_cols)

X[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())

X = pd.get_dummies(X, columns=categorical_cols)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

stratify_col = y_resampled
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=stratify_col)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

def build_nn_model(hp):
    input_layer = Input(shape=(X_train_scaled.shape[1],))
    x = Dense(hp.Int('units_1', min_value=128, max_value=512, step=64), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(input_layer)
    x = BatchNormalization()(x)
    x = Dropout(hp.Float('dropout_1', min_value=0.2, max_value=0.5, step=0.1))(x)
    x_residual = x

    x = Dense(hp.Int('units_2', min_value=128, max_value=512, step=64), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(hp.Float('dropout_2', min_value=0.2, max_value=0.5, step=0.1))(x)

    if x.shape[-1] != x_residual.shape[-1]:
        x_residual = Dense(x.shape[-1], kernel_initializer='he_normal')(x_residual)

    x = add([x, x_residual])
    x = Activation('relu')(x)

    x_expanded = Lambda(lambda x: tf.expand_dims(x, axis=1))(x)

    attention = Attention()([x_expanded, x_expanded])

    attention = Lambda(lambda x: tf.squeeze(x, axis=1))(attention)

    x = add([x, attention])
    x = Activation('relu')(x)

    x = Dense(hp.Int('units_3', min_value=64, max_value=256, step=64), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(hp.Float('dropout_3', min_value=0.2, max_value=0.5, step=0.1))(x)
    output_layer = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')), loss='binary_crossentropy', metrics=['accuracy'])
    return model

with open('/kaggle/input/notamodel/other/default/1/best_nn_model/trial_00/trial.json') as f:
    best_hps_data = json.load(f)

hp = HyperParameters()
hp.Fixed('units_1', best_hps_data['hyperparameters']['values']['units_1'])
hp.Fixed('dropout_1', best_hps_data['hyperparameters']['values']['dropout_1'])
hp.Fixed('units_2', best_hps_data['hyperparameters']['values']['units_2'])
hp.Fixed('dropout_2', best_hps_data['hyperparameters']['values']['dropout_2'])
hp.Fixed('units_3', best_hps_data['hyperparameters']['values']['units_3'])
hp.Fixed('dropout_3', best_hps_data['hyperparameters']['values']['dropout_3'])
hp.Fixed('learning_rate', best_hps_data['hyperparameters']['values']['learning_rate'])

nn_model = build_nn_model(hp)

early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5)

nn_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr]
)

nn_model.save('/kaggle/working/nn_model.h5')

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}
rf_random_search = RandomizedSearchCV(rf_model, rf_param_grid, n_iter=10, scoring='roc_auc', cv=3, random_state=42, n_jobs=-1)
rf_random_search.fit(X_train, y_train)
rf_model_best = rf_random_search.best_estimator_

rf_model_best_path = '/kaggle/working/rf_model_best.joblib'
import joblib
joblib.dump(rf_model_best, rf_model_best_path)

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

gb_model_best_path = '/kaggle/working/gb_model_best.joblib'
joblib.dump(gb_model, gb_model_best_path)

nn_pred_prob = nn_model.predict(X_val_scaled).flatten()
rf_pred_prob = rf_model_best.predict_proba(X_val)[:, 1]
gb_pred_prob = gb_model.predict_proba(X_val)[:, 1]

combined_features = np.vstack((nn_pred_prob, rf_pred_prob, gb_pred_prob)).T
stacking_meta_model = GradientBoostingClassifier(n_estimators=50, learning_rate=0.05, max_depth=3, random_state=42)
stacking_meta_model.fit(combined_features, y_val)

stacking_meta_model_path = '/kaggle/working/stacking_meta_model.joblib'
joblib.dump(stacking_meta_model, stacking_meta_model_path)

final_pred_prob = stacking_meta_model.predict_proba(combined_features)[:, 1]
roc_auc = roc_auc_score(y_val, final_pred_prob)
print(f'Validation ROC AUC Score (Ensemble): {roc_auc}')

test_df = pd.read_csv('/kaggle/input/ds-310-project-2-fall-2024/test_data.csv')
X_test = test_df.drop(['Name'], axis=1)

X_test[numerical_cols] = X_test[numerical_cols].fillna(X[numerical_cols].median())

X_test = pd.get_dummies(X_test, columns=categorical_cols)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

X_test_scaled = scaler.transform(X_test)

nn_test_pred_prob = nn_model.predict(X_test_scaled).flatten()
rf_test_pred_prob = rf_model_best.predict_proba(X_test)[:, 1]
gb_test_pred_prob = gb_model.predict_proba(X_test)[:, 1]

combined_test_features = np.vstack((nn_test_pred_prob, rf_test_pred_prob, gb_test_pred_prob)).T
test_final_pred_prob = stacking_meta_model.predict_proba(combined_test_features)[:, 1]

test_df['Class'] = test_final_pred_prob

test_df[['Name', 'Class']].to_csv('/kaggle/working/ubmission.csv', index=False)
print('Test predictions saved to /kaggle/working/submission.csv')
